In [106]:
import pandas as pd 
import exifread
import uuid
import requests_cache
import openmeteo_requests

from tqdm.auto import tqdm 
from retry_requests import retry
from datetime import datetime

In [107]:
# test array of photo paths

photo_paths = [
    "images/iceland_east_coastline.jpg",
    "images/iceland_west_coastline.jpg",
]

def dms_to_decimal(dms, ref):
    degrees = float(dms.values[0].num) / float(dms.values[0].den)
    minutes = float(dms.values[1].num) / float(dms.values[1].den)
    seconds = float(dms.values[2].num) / float(dms.values[2].den)
    decimal = round(degrees + minutes / 60 + seconds / 3600, 2)
    if ref in ("S", "W"):
        decimal = -decimal
    return decimal

def format_dt(exif_dt):
    return datetime.strptime(str(exif_dt), "%Y:%m:%d %H:%M:%S")

def generate_photo_id()-> str:
    return uuid.uuid4().hex[:12]

def extract_lat_lon_time(photo_path):
    with open(photo_path, "rb") as f:
        tags = exifread.process_file(f, details=False)

    if (
        "GPS GPSLatitude" in tags
        and "GPS GPSLongitude" in tags
        and "EXIF DateTimeOriginal" in tags
    ):
        lat = dms_to_decimal(tags["GPS GPSLatitude"], str(tags["GPS GPSLatitudeRef"]))
        lon = dms_to_decimal(tags["GPS GPSLongitude"], str(tags["GPS GPSLongitudeRef"]))
        dt = format_dt(tags["EXIF DateTimeOriginal"])
        photo_id = generate_photo_id()
    else:
        lat, lon = 90.0, 0.0
        dt = datetime(2020, 1, 1, 23, 0, 0)
        photo_id = generate_photo_id()

    return {
        "id": photo_id,
        "photo_path": photo_path,
        "datetime_taken": dt,
        "lat": lat,
        "lon": lon,
    }

rows = [extract_lat_lon_time(path) for path in photo_paths]

photo_data_df = pd.DataFrame(rows, columns=["id", "photo_path", "datetime_taken", "lat", "lon"])

photo_data_df


,id,photo_path,datetime_taken,lat,lon
0,f4842c626f58,images/iceland_east_coastline.jpg,2017-09-26 15:55:11,64.45,-14.51
1,f58d429bed41,images/iceland_west_coastline.jpg,2017-09-30 12:44:46,64.95,-23.42


In [108]:
photo_data_df["hour"] = photo_data_df['datetime_taken'].dt.round('h').dt.hour

photo_data_df.head()

,id,photo_path,datetime_taken,lat,lon,hour
0,f4842c626f58,images/iceland_east_coastline.jpg,2017-09-26 15:55:11,64.45,-14.51,16
1,f58d429bed41,images/iceland_west_coastline.jpg,2017-09-30 12:44:46,64.95,-23.42,13


In [109]:
# add weather columns before adding weather data - fill with None

weather_cols = ["temp_f", "cloud_cover"]
for col in weather_cols:
    if col not in photo_data_df.columns:
        photo_data_df[col] = None
        
print(f"Loaded {len(photo_data_df):,} records")

Loaded 2 records


In [110]:
# add weather data to dataframe 

# Setup client 
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)


def get_weather(lat, lon, date_taken, hour=0):
    lat = round(float(lat), 2)
    lon = round(float(lon), 2)
    date_taken = pd.to_datetime(date_taken).strftime("%Y-%m-%d")
    hour = int(hour) if pd.notna(hour) else 0
    hour = hour if 0 <= hour <= 23 else 0

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": date_taken,
        "end_date": date_taken,
        "hourly": ["temperature_2m", "cloud_cover"],
        "temperature_unit": "fahrenheit",
        "timezone": "auto",
    }

    try:
        responses = openmeteo.weather_api(
            "https://archive-api.open-meteo.com/v1/archive", params=params
        )
        hourly = responses[0].Hourly()
        temp = hourly.Variables(0).ValuesAsNumpy()
        cloud = hourly.Variables(1).ValuesAsNumpy()

        return {
            "temp_f": round(float(temp[hour]), 1) if len(temp) > hour else None,
            "cloud_cover": round(float(cloud[hour]), 1) if len(cloud) > hour else None,
        }
    except Exception as e:
        print(f"Weather API error ({lat}, {lon}, {date_taken} {hour}): {e}")
        return {"temp_f": None, "cloud_cover": None}



In [111]:
for idx, row in tqdm(photo_data_df.iterrows(), total=len(photo_data_df)):
    try:
        weather = get_weather(
            lat=row["lat"],
            lon=row["lon"],
            date_taken=row["datetime_taken"].strftime("%Y-%m-%d"),
            hour=row["hour"]
        )
        for key, value in weather.items():
            photo_data_df.at[idx, key] = value

    except Exception as e:
        print(f"Error at row {idx}: {e}")

photo_data_df


  0%|          | 0/2 [00:00<?, ?it/s]

,id,photo_path,datetime_taken,lat,lon,hour,temp_f,cloud_cover
0,f4842c626f58,images/iceland_east_coastline.jpg,2017-09-26 15:55:11,64.45,-14.51,16,52.2,100.0
1,f58d429bed41,images/iceland_west_coastline.jpg,2017-09-30 12:44:46,64.95,-23.42,13,48.3,98.0
